# EVA-CLM: T4 pre-A100 notebook (dry-run prod + mid-scale A/B)

Бюджет: Colab T4 (~5ч). Setup — как в основном A100-ноутбуке (Drive + клон GitHub).
Все артефакты пишутся на Drive в logs/t4: итоговый JSON атомарно перезаписывается
на каждом eval (и минимум раз в 200 шагов), progress.txt дописывается по ходу —
смерть VM не теряет прогон (ckpt каждого арма появляется в конце арма).
1. **dry-run прода** (5-10 мин): влезает ли D=2560/24L в 16GB (gc=True+AMP) и s/step.
   Порог: s/step > 30 -> прод на T4 не тратить.
2. **mid-scale A/B** (~280 мин: ~140 на арм): bounded_residual=True vs False, D=512/8L, реальные данные.

Runtime -> Change runtime type -> T4 GPU. Затем ячейки 1-8 по порядку.


In [ ]:
# 1. Mount Drive (как в A100-ноутбуке)
import os, sys, glob, json, subprocess
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/eva_clm'   # @param
DATA_DIR   = os.path.join(DRIVE_ROOT, 'data')
print('DATA_DIR =', DATA_DIR)


In [ ]:
# 2. Clone/pull код с GitHub (как в A100-ноутбуке)
DST = '/content/eva_clm'
if not os.path.exists(DST):
    print('Cloning from GitHub...')
    subprocess.run(['git', 'clone', 'https://github.com/BlackCatSpb/EVA-CLM.git', DST], check=True)
else:
    print('Force-pulling latest...')
    subprocess.run(['git', '-C', DST, 'fetch', 'origin', 'master'], check=True)
    subprocess.run(['git', '-C', DST, 'reset', '--hard', 'origin/master'], check=True)
print(subprocess.run(['git', '-C', DST, 'log', '--oneline', '-1'],
                     capture_output=True, text=True).stdout)
sys.path.insert(0, DST)
os.chdir(DST)
print('cwd:', os.getcwd())


In [ ]:
# 3. Verify GPU & allocator
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0),
          'VRAM: %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))
    print('bf16 supported:', torch.cuda.is_bf16_supported())
    try:
        torch.cuda.memory._set_allocator_settings('expandable_segments:True')
        print('allocator: expandable_segments=ON')
    except Exception as e:
        print('allocator:', type(e).__name__)
else:
    print('CUDA не доступна — переключи Runtime на T4')


In [ ]:
# 4. Данные: стримы в DATA_DIR (fallback: репо/wb)
import glob, os
files = []
for pat in ('token_stream_*_eos.bin', 'token_stream_*_clean.bin', 'token_stream_*.bin'):
    files = sorted(glob.glob(os.path.join(DATA_DIR, pat)))
    if files:
        break
if not files:
    files = sorted(glob.glob(os.path.join(DST, 'wb', 'token_stream_*.bin')))
    DATA_DIR = os.path.join(DST, 'wb')
assert files, 'нет token_stream_*.bin ни в DATA_DIR, ни в репо/wb'
print('data dir:', DATA_DIR, '| streams:', len(files))
for f in files[:6]:
    print(' ', os.path.basename(f), round(os.path.getsize(f) / 1e6, 1), 'MB')
war = [f for f in files if 'WAR' in os.path.basename(f).upper()]
HOLDOUT = os.path.basename(war[0]) if war else os.path.basename(files[-1])
print('HOLDOUT =', HOLDOUT)


In [ ]:
# 5. DRY-RUN прода (5-10 мин): peak VRAM, s/step, b_flow/b_drift
import os
os.makedirs(os.path.join(DRIVE_ROOT, 'logs', 't4'), exist_ok=True)
!python scripts/t4_bounded_ab.py --dry-run --data-dir {DATA_DIR} --out-dir {DRIVE_ROOT}/logs/t4
import glob, json
OUT_T4 = os.path.join(DRIVE_ROOT, 'logs', 't4')
cands = sorted(glob.glob(os.path.join(OUT_T4, 't4_dryrun_*.json')))
p = cands[-1] if cands else None
if p:
    d = json.load(open(p, encoding='utf-8'))
    print('=== DRY-RUN', p, '===')
    print(json.dumps(d, indent=1, ensure_ascii=False)[:3000])
    s = d.get('s_per_step') or d.get('sec_per_step')
    if s and float(s) > 30:
        print('>>> s/step > 30: прод на T4 НЕ тратить')
else:
    print('нет t4_dryrun_*.json в', OUT_T4, '— смотри консоль выше')


In [ ]:
# 6. MID-SCALE A/B: bounded_residual=True vs False (~280 мин, ~140 на арм)
import os
os.makedirs(os.path.join(DRIVE_ROOT, 'logs', 't4'), exist_ok=True)
!python scripts/t4_bounded_ab.py --ab --steps 100000 --max-minutes 200 --data-dir {DATA_DIR} --out-dir {DRIVE_ROOT}/logs/t4 --holdout {HOLDOUT}


In [ ]:
# 7. Результаты из Drive: JSON + progress.txt + txt/log (пришли их мне для разбора)
import glob, os, json
OUT_T4 = os.path.join(DRIVE_ROOT, 'logs', 't4')
prog = os.path.join(OUT_T4, 'progress.txt')
if os.path.exists(prog):
    lines = open(prog, encoding='utf-8', errors='replace').read().splitlines()
    print('=' * 20, prog, f'({len(lines)} lines, tail 40)')
    print('\n'.join(lines[-40:]))
for f in sorted(glob.glob(os.path.join(OUT_T4, '*'))):
    if os.path.basename(f) == 'progress.txt':
        continue
    print('=' * 20, f)
    if f.endswith('.json'):
        try:
            print(json.dumps(json.load(open(f, encoding='utf-8')), indent=1, ensure_ascii=False)[:4000])
        except Exception as e:
            print('json error:', e)
    elif f.endswith(('.txt', '.log')):
        print(open(f, encoding='utf-8', errors='replace').read()[:4000])
